In [ ]:
import numpy as np

In [ ]:
errors=[]
for latent_dim in [1,2,4,8,16,32,64]:
    for step_size in [1,2,4,8,16,32,64]:
        try:
            reconstruction_train=np.load(f'reconstructions_train_{latent_dim}_{step_size}.npy')
            train=np.load(f'X_train.npy')
            error=np.mean(np.square(train - reconstruction_train), axis=1)
            errors.append([latent_dim,step_size,error])
        except Exception as e:
            # This prints the loop variables AND the specific error message
            print(e)
        

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations

# --- Part 0: Load MDR Data ---
print(f"Loading MDR data from 'analysis_results_flat.csv'...")
try:
    df_mdr = pd.read_csv('analysis_results_flat.csv')
    
    # Create a lookup dictionary: {(latent_dim, step_size): mdr_value}
    # We use the exact column names you provided.
    mdr_lookup = df_mdr.set_index(
        ['latent_dim', 'step_size']
    )['maha_dist_ratio_95_mean'].to_dict()
    
    print(f"Successfully created MDR lookup for {len(mdr_lookup)} models.")

except Exception as e:
    print(f"Error loading or processing 'analysis_results_flat.csv': {e}")
    print("Cannot proceed without MDR data. Exiting.")
    mdr_lookup = {} # Create empty dict to avoid later errors if you run this

# --- Part 1: Process Errors and Find Top-K Indices ---

# Define your K values
K_values = [50, 100, 200]

# Use a dictionary to store the SET of top-K indices for each model
# Structure: {(latent_dim, step_size): {50: {idx1, ...}, 100: {idx1, ...}, 200: {idx1, ...}}}
top_k_indices = {}

latent_dims = [1, 2, 4, 8, 16, 32, 64]
step_sizes = [1, 2, 4, 8, 16, 32, 64]

print("Processing models and finding top-K anomalies...")
for latent_dim in latent_dims:
    for step_size in step_sizes:
        try:
            # Load the data
            reconstruction_train = np.load(f'reconstructions_train_{latent_dim}_{step_size}.npy')
            train = np.load(f'X_train.npy')
            
            # Calculate MSE for each sample
            # error is our "anomaly score"
            error = np.mean(np.square(train - reconstruction_train), axis=1)
            
            # Get the indices that would sort the error array (ascending)
            # We use argsort as it's efficient and gives us the indices
            sorted_indices = np.argsort(error)
            
            model_key = (latent_dim, step_size)
            top_k_indices[model_key] = {}
            
            # Find the top-K indices for each K
            for k in K_values:
                # Get the last K indices, which correspond to the highest errors
                top_k_set = set(sorted_indices[-k:])
                top_k_indices[model_key][k] = top_k_set
                
        except Exception as e:
            # We check if the key is in our lookup, otherwise it's just a missing file
            if (latent_dim, step_size) in mdr_lookup:
                print(f"Error loading/processing {latent_dim}_{step_size} .npy files: {e}")
            # If not in mdr_lookup, it's just a model combo you don't have, which is fine.

print(f"Successfully processed {len(top_k_indices)} models.")

# --- Part 2: Calculate Jaccard Index for All Model Pairs ---

# The Jaccard Index is: |A intersection B| / |A union B|

jaccard_results = []
model_keys = list(top_k_indices.keys())

# Iterate over all unique pairs of models
for model_A_key, model_B_key in combinations(model_keys, 2):
    
    # Compare the pairs for each K
    for k in K_values:
        set_A = top_k_indices[model_A_key][k]
        set_B = top_k_indices[model_B_key][k]
        
        intersection = len(set_A.intersection(set_B))
        union = len(set_A.union(set_B))
        
        if union == 0:
            # Handle edge case of two empty sets (shouldn't happen if K > 0)
            jaccard_score = 1.0 if intersection == 0 else 0.0
        else:
            jaccard_score = intersection / union
            
        # Store the result
        jaccard_results.append({
            'model_A': model_A_key,
            'model_B': model_B_key,
            'mdr_A': mdr_lookup.get(model_A_key),    # <-- NEW
            'mdr_B': mdr_lookup.get(model_B_key),    # <-- NEW
            'K': k,
            'jaccard_index': jaccard_score,
            'overlap_count': intersection
        })

# --- Part 3: Display Results using Pandas ---

print("\n--- Jaccard Index Results ---")

if jaccard_results:
    # Convert results to a DataFrame for easy viewing
    df_results = pd.DataFrame(jaccard_results)
    
    # Set model keys (tuples) as readable strings
    df_results['model_A'] = df_results['model_A'].apply(lambda x: f"LD={x[0]}, SS={x[1]}")
    df_results['model_B'] = df_results['model_B'].apply(lambda x: f"LD={x[0]}, SS={x[1]}")
    
    # Re-order columns to be more readable
    cols = ['model_A', 'model_B', 'mdr_A', 'mdr_B', 'K', 'jaccard_index', 'overlap_count']
    # Filter out any columns not present, just in case
    cols = [c for c in cols if c in df_results.columns]
    df_results = df_results[cols]
    
    print(df_results)

    # --- Save the final DataFrame to a new CSV ---
    try:
        df_results.to_csv('jaccard_results_with_mdr.csv', index=False) # <-- NEW
        print("\nSaved full results to 'jaccard_results_with_mdr.csv'")
    except Exception as e:
        print(f"\nError saving results to CSV: {e}")

    # You can also get the average Jaccard index for each K
    print("\n--- Average Jaccard Index per K ---")
    print(df_results.groupby('K')['jaccard_index'].mean())
    
    # Or create a pivot table for one K value (e.g., K=100)
    print("\n--- Pivot Table for K=100 ---")
    df_k100 = df_results[df_results['K'] == 100]
    try:
        pivot_df = df_k100.pivot_table(index='model_A', columns='model_B', values='jaccard_index')
        print(pivot_df)
    except Exception as e:
        print(f"Could not create pivot table: {e}")
        print("Displaying K=100 results as a list:")
        print(df_k100)

else:
    print("No Jaccard results were calculated. Check if models were processed.")

In [ ]:
df_k100.to_csv('Jaccard.csv')